# 第 12 节：REINFORCE 从零实现

## 📍 位置
策略梯度推导 (11) → **REINFORCE (12)** → Baseline (13) → ...

## 🎯 学习目标
1. 从零实现 REINFORCE 算法
2. 理解 loss = -log_prob * return 的含义
3. 理解为什么负号对应梯度上升
4. 在 CartPole 上训练并分析结果
5. 对比有无 baseline 的方差差异

## 1. REINFORCE 算法回顾

### 更新规则
$$\theta \leftarrow \theta + \alpha \cdot \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t$$

### PyTorch 实现要点
```python
# 注意负号！PyTorch 做梯度下降，我们想要梯度上升
loss = -(log_prob * G_t).mean()
loss.backward()
optimizer.step()
```

### 完整流程
1. 用当前策略 π_θ 采集一条完整 episode
2. 计算每个时间步的折扣回报 G_t
3. 计算 loss = -Σ_t log π_θ(a_t|s_t) · G_t
4. 梯度下降（等价于策略梯度上升）
5. 重复

In [ ]:
%matplotlib inline
import sys; sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from rl_course.utils.seeding import set_seed; set_seed(42)
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque
import os

FIG_DIR = 'outputs/figures'
os.makedirs(FIG_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 2. 从零实现 REINFORCE

In [ ]:
class REINFORCE:
    """REINFORCE (Monte Carlo Policy Gradient) 从零实现

    核心公式:
        ∇J(θ) = E[∇ log π(a|s) · G]
        loss = -mean(log_prob * G)
    """

    def __init__(self, state_dim, n_actions, hidden_dim=128, lr=1e-2, gamma=0.99):
        self.gamma = gamma
        self.n_actions = n_actions

        # 策略网络
        self.policy_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        ).to(DEVICE)

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.episode_data = []  # 存储 (log_prob, reward)

    def act(self, state, train=True):
        """根据策略采样动作"""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)  # (1, state_dim)
        logits = self.policy_net(state_t)  # (1, n_actions)
        probs = torch.softmax(logits, dim=-1)

        if not train:
            return torch.argmax(probs, dim=-1).item()

        # 采样动作
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)  # (1,)

        self.episode_data.append((log_prob, None))  # reward 稍后填入
        return action.item()

    def store_reward(self, reward):
        """存储最近一步的奖励"""
        log_prob, _ = self.episode_data[-1]
        self.episode_data[-1] = (log_prob, reward)

    def update(self):
        """在一个 episode 结束后更新参数"""
        # 计算折扣回报 G_t（从后向前）
        returns = []
        G = 0.0
        for _, reward in reversed(self.episode_data):
            G = reward + self.gamma * G
            returns.insert(0, G)

        returns = torch.FloatTensor(returns).to(DEVICE)  # (T,)

        # 标准化回报（减少方差，加速训练）
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # 计算策略梯度 loss
        policy_loss = []
        for (log_prob, _), G_t in zip(self.episode_data, returns):
            policy_loss.append(-log_prob * G_t)  # 负号！梯度上升 → 梯度下降

        policy_loss = torch.cat(policy_loss).mean()

        # 反向传播
        self.optimizer.zero_grad()
        policy_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=1.0)
        self.optimizer.step()

        # 清空 episode 数据
        self.episode_data = []

        return policy_loss.item(), returns.mean().item()

print("✅ REINFORCE 类定义完成")

## 3. 训练 REINFORCE on CartPole

In [ ]:
# 创建环境
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f"CartPole: state_dim={state_dim}, n_actions={n_actions}")

# 创建 agent
agent = REINFORCE(state_dim, n_actions, hidden_dim=128, lr=1e-2, gamma=0.99)

# 训练循环
n_episodes = 500  # 快速模式
episode_returns = []
losses = []

for ep in range(n_episodes):
    state, _ = env.reset()
    done = False
    ep_return = 0

    while not done:
        action = agent.act(state, train=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.store_reward(reward)
        state = next_state
        ep_return += reward

    # Episode 结束，更新策略
    loss, avg_return = agent.update()
    episode_returns.append(ep_return)
    losses.append(loss)

    if (ep + 1) % 100 == 0:
        recent = np.mean(episode_returns[-50:])
        print(f"Episode {ep+1:4d} | Avg Return (last 50): {recent:6.1f}")

print(f"\\n训练完成！最终 50 episode 平均回报: {np.mean(episode_returns[-50:]):.1f}")


## 4. 训练曲线分析

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Episode 回报
ax1.plot(episode_returns, alpha=0.3, linewidth=0.5, color='steelblue')
window = 20
if len(episode_returns) > window:
    smoothed = np.convolve(episode_returns, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(episode_returns)), smoothed, linewidth=2, color='red')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Return')
ax1.set_title('REINFORCE on CartPole'); ax1.grid(True, alpha=0.3)
ax1.axhline(y=500, color='green', linestyle='--', alpha=0.5, label='Max (500)')
ax1.legend()

# Loss
ax2.plot(losses, linewidth=0.5, alpha=0.7, color='coral')
ax2.set_xlabel('Episode'); ax2.set_ylabel('Policy Loss')
ax2.set_title('REINFORCE Loss'); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/12_reinforce_cartpole.png', dpi=100); plt.close()
print("✅ 训练曲线已保存")

## 5. 智能体演示视频

In [ ]:
# 录制智能体运行
from rl_course.visualization.video import record_episode

# 包装为 policy function
def policy_fn(state):
    return agent.act(state, train=False)

# 创建新的 env（render_mode="rgb_array"）
env_render = gym.make("CartPole-v1", render_mode="rgb_array")
env_render.reset()
record_episode(env_render, policy_fn, filepath='outputs/videos/12_reinforce_cartpole.gif', fps=20, max_steps=500)
env_render.close()
print("✅ 演示视频已保存")

## 6. 常见问题分析

### 为什么 REINFORCE 不稳定？
1. **高方差**：$G_t$ 累积了所有未来步骤的随机性
2. **样本效率低**：每条 episode 只做一次梯度更新
3. **步长敏感**：学习率太大→策略崩溃；太小→收敛慢

### Loss 震荡意味什么？
- Policy loss 的绝对值没有意义（取决于 return 的 scale）
- 关键是 episode return 是否稳定上升

### 为什么需要 return 标准化？
- 不标准化：不同 episode 的 return 量级差异大，梯度不稳定
- 标准化后：梯度更平稳，收敛更快

## 7. 总结

REINFORCE 是策略梯度最基础的形式：
- **优点**：简单、无偏、易实现
- **缺点**：高方差、必须等 episode 结束

下一步：用 **baseline** 减少方差 → 用 **bootstrapping** 做到在线更新 (Actor-Critic)

## 8. 练习
1. 去掉 return 标准化，观察训练是否还能收敛
2. 比较 γ=0.9, 0.99, 1.0 的效果
3. 将 hidden_dim 从 128 改为 32，观察影响
4. 在 Acrobot-v1 环境上测试 REINFORCE

---
*下一节：[13_baseline_and_advantage.ipynb](13_baseline_and_advantage.ipynb)*